# HealthConnect Clinic — Week 8 Final Analytics & Decision Support

**Track:** Data Analytics  
**Project:** HealthConnect Clinic  
**Project Title:** Improving Patient Appointment Attendance and Healthcare Support Using Data and AI

### Week 8 Focus
Final integration, validation, decision support and presentation readiness.

This notebook continues the HealthConnect Analytics work. It does not introduce a new project or dataset. The purpose is to provide a reproducible analytical evidence trail for the final Week 8 package.

## 1. Executive Summary

The final Analytics work validates the core HealthConnect appointment KPIs and the major no-show patterns established during Weeks 5–7.

The cleaned dataset contains **5,000 appointments**. Final validated KPIs are:

- No-Show Rate: **48.46%**
- Attendance Rate: **46.28%**
- Average Booking Lead Time: **29.64 days**
- Reminder Coverage: **72.68%**

## 2. Business Scenario

HealthConnect Clinic wants to reduce missed appointments, improve attendance, make better use of available appointment slots and provide more effective administrative support.

The Analytics question is:

> **How can HealthConnect Clinic use appointment data to understand no-show patterns and support better attendance-related decisions?**

The final Week 8 Analytics focuses on validated evidence rather than introducing new exploratory scope.

## 3. Project Continuity and Scope

### Week 4
Problem understanding, resource review and solution planning.

### Week 5
Data cleaning, exploratory analysis, initial findings and Power BI development.

### Week 6
Deeper analysis, combined-variable analysis, dashboard refinement and cross-track analytical support.

### Week 7
Testing, refinement and end-to-end validation.

### Week 8
Final integration, decision support and presentation readiness.

### Week 8 analytical scope

This notebook:

1. reloads the cleaned dataset;
2. confirms data dimensions and quality;
3. validates final KPIs;
4. reproduces important Week 7 findings;
5. analyzes selected variables together;
6. validates supporting variables;
7. reviews dashboard figures;
8. documents business insights and recommendations;and
9. records analytical limitations

## 4. Week 7 → Week 8 Transition

### Main Week 7 Output
Validated Analytics calculations, findings and Power BI dashboard refinements.

### Most Important Testing Result
The principal KPIs and selected Week 6 findings were reproducible from the cleaned dataset.

### Major Issue Identified
Analytics and Data Science use different denominators for no-show rate. This is a definition difference, not a discrepancy in the number of no-shows.

### Refinement
Combined-variable findings, dashboard visuals, slicers and calculations were retested.

### Component Ready
The Analytics evidence and dashboard are ready for final integration and presentation.

## 5. Dataset and Analytical Scope

Primary analytical variables include:

- appointment outcome;
- booking lead time;
- previous no-show history;
- reminder status/channel;
- distance;
- waiting time;
- age;
- appointment date;
- appointment type.

The Week 8 focus is on validated findings from previous weeks rather than a new exploratory analysis.

In [1]:
## Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
# Load dataset
file_path = "HealthConnect_Appointment_Data_Cleaned.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully.")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

Dataset loaded successfully.
Rows: 5,000
Columns: 19


## 6. Dataset Size Confirmation

The expected final analytical dataset contains **5,000 rows and 19 columns**.

In [3]:
# Confirm dataset size
expected_rows = 5000
expected_columns = 19

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

assert df.shape == (expected_rows, expected_columns), (
    f"Unexpected dataset shape: {df.shape}"
)

print("PASS: Dataset shape matches the expected 5,000 × 19 structure.")

Rows: 5000
Columns: 19
PASS: Dataset shape matches the expected 5,000 × 19 structure.


## 7. Data Quality Confirmation

Week 4 to Week 7 data-quality work identified:

- no duplicate records;
- reminder_channel missing for appointments where a reminder was not sent;
- 90 missing distance values;
- 60 missing waiting-time values.

The reminder-channel missingness is treated as structural where reminder_sent = No, rather than automatically treating it as a data-entry error.

In [4]:
# Data quality check
duplicate_count = df.duplicated().sum()
missing_summary = df.isna().sum().sort_values(ascending=False)

print("Duplicate rows:", duplicate_count)
print("\nMissing values:")
display(missing_summary[missing_summary > 0])

assert duplicate_count == 0, "Duplicate rows detected."

print("\nPASS: No duplicate records detected.")

Duplicate rows: 0

Missing values:


reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
dtype: int64


PASS: No duplicate records detected.


## 8. Date and Lead-Time Preparation

Date fields are converted to datetime where present.

The existing booking_lead_days field is used as the analytical lead-time measure.

In [5]:
# Conversion to datetime datatypes
date_columns = [c for c in ["booking_date", "appointment_date"] if c in df.columns]

for col in date_columns:
    df[col] = pd.to_datetime(df[col], errors="coerce")

if "booking_lead_days" in df.columns:
    df["booking_lead_days"] = pd.to_numeric(df["booking_lead_days"], errors="coerce")

print(df[date_columns].dtypes if date_columns else "No date columns found.")

booking_date        datetime64[ns]
appointment_date    datetime64[ns]
dtype: object


In [6]:
if {"booking_date", "appointment_date"}.issubset(df.columns):
    df["calculated_lead_days"] = (
        df["appointment_date"] - df["booking_date"]
    ).dt.days

    comparison = df[["booking_lead_days", "calculated_lead_days"]].dropna()
    comparison["difference"] = (
        comparison["booking_lead_days"] - comparison["calculated_lead_days"]
    )

    print("Maximum absolute lead-time difference:",
          comparison["difference"].abs().max())

    display(comparison["difference"].value_counts().head())

    assert comparison["difference"].abs().max() == 0, (
        "Lead-time calculation does not fully match the stored lead-time field."
    )

    print("PASS: Independently calculated lead time matches booking_lead_days.")

Maximum absolute lead-time difference: 0


difference
0    5000
Name: count, dtype: int64

PASS: Independently calculated lead time matches booking_lead_days.


## 9. Final KPI Validation

The Week 8 final KPI scope contains five measures:

1. Total Appointments
2. No-Show Rate
3. Attendance Rate
4. Average Booking Lead Time
5. Reminder Coverage

In [7]:
# Detect likely outcome column
possible_outcome_cols = [
    c for c in df.columns
    if c.lower() in {"appointment_status", "status", "outcome", "appointment_outcome"}
]

print("Possible outcome columns:", possible_outcome_cols)

outcome_col = possible_outcome_cols[0] if possible_outcome_cols else "appointment_outcome"

print("Using outcome column:", outcome_col)
print(df[outcome_col].value_counts(dropna=False))

Possible outcome columns: ['appointment_outcome']
Using outcome column: appointment_outcome
appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64


In [8]:
total_appointments = len(df)

outcome_text = df[outcome_col].astype(str).str.strip().str.lower()

no_show_count = (outcome_text == "no-show").sum()
attended_count = (outcome_text == "attended").sum()

no_show_rate = no_show_count / total_appointments * 100
attendance_rate = attended_count / total_appointments * 100

avg_lead_time = df["booking_lead_days"].mean()

# Reminder coverage is based on whether a reminder was sent.
possible_reminder_sent_cols = [
    c for c in df.columns if c.lower() == "reminder_sent"
]
reminder_sent_col = possible_reminder_sent_cols[0] if possible_reminder_sent_cols else None

if reminder_sent_col:
    reminder_text = df[reminder_sent_col].astype(str).str.strip().str.lower()
    reminder_coverage = (reminder_text == "yes").mean() * 100
else:
    reminder_coverage = np.nan

final_kpis = pd.DataFrame({
    "KPI": [
        "Total Appointments",
        "No-Show Rate",
        "Attendance Rate",
        "Average Booking Lead Time",
        "Reminder Coverage"
    ],
    "Value": [
        total_appointments,
        no_show_rate,
        attendance_rate,
        avg_lead_time,
        reminder_coverage
    ],
    "Expected": [
        5000,
        48.46,
        46.28,
        29.64,
        72.68
    ]
})

final_kpis["Difference"] = (final_kpis["Value"] - final_kpis["Expected"]).round(2)
final_kpis.loc[final_kpis["Difference"].abs() < 0.01, "Difference"] = 0.00
display(final_kpis)

,KPI,Value,Expected,Difference
0,Total Appointments,"5,000.00","5,000.00",0.00
1,No-Show Rate,48.46,48.46,0.00
2,Attendance Rate,46.28,46.28,0.00
3,Average Booking Lead Time,29.64,29.64,0.00
4,Reminder Coverage,72.68,72.68,0.00


In [9]:
# Validate against Week 7 final values using a small tolerance for rounding.
assert total_appointments == 5000
assert abs(no_show_rate - 48.46) < 0.01
assert abs(attendance_rate - 46.28) < 0.01
assert abs(avg_lead_time - 29.64) < 0.01
assert abs(reminder_coverage - 72.68) < 0.01

print("PASS: All final KPI values match the validated Week 7 values within rounding tolerance.")

PASS: All final KPI values match the validated Week 7 values within rounding tolerance.


## 10. Appointment Outcome Distribution

In [10]:
# Calculate appointment outcome distribution

outcome_summary = (
    df["appointment_outcome"]
    .value_counts()
    .rename_axis("Appointment Status")
    .reset_index(name="Count")
)

outcome_summary["Percentage"] = (
    outcome_summary["Count"] / total_appointments * 100
)

display(outcome_summary)


,Appointment Status,Count,Percentage
0,No-Show,2423,48.46
1,Attended,2314,46.28
2,Cancelled,263,5.26


## 11. Lead-Time Analysis

Week 5 established an increasing no-show pattern across lead-time groups.

The Week 8 notebook reproduces this finding for validation.

In [11]:
# Create validated lead-time groups

lead_time_bins = [-1, 7, 14, 30, 45, 60]
lead_time_labels = [
    "0–7",
    "8–14",
    "15–30",
    "31–45",
    "46–60"
]

df["lead_time_group"] = pd.cut(
    df["booking_lead_days"],
    bins=lead_time_bins,
    labels=lead_time_labels
)

lead_time_summary = (
    df.groupby("lead_time_group", observed=False)
      .agg(
          appointments=("appointment_outcome", "size"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
)

lead_time_summary["no_show_rate"] = (
    lead_time_summary["no_shows"]
    / lead_time_summary["appointments"]
    * 100
)

display(lead_time_summary)


,appointments,no_shows,no_show_rate
lead_time_group,,,
0–7,640,178,27.81
8–14,602,202,33.55
15–30,1333,576,43.21
31–45,1258,677,53.82
46–60,1167,790,67.69


### Validated finding

The no-show rate increases across the booking lead-time groups, with the highest observed rate occurring among appointments booked 46–60 days ahead.

This finding was already established and validated during Week 7. Week 8 reproduces it as part of the final analytical evidence.


## 12. Previous No-Show History

Previous no-show history was identified as another important variable associated with future appointment attendance.

For the final analysis, previous no-shows are grouped into:

- 0 previous no-shows
- 1 previous no-show
- 2+ previous no-shows

In [12]:
# Create previous no-show history groups

df["previous_no_show_group"] = pd.cut(
    df["previous_no_shows"],
    bins=[-1, 0, 1, np.inf],
    labels=["0", "1", "2+"]
)

previous_summary = (
    df.groupby("previous_no_show_group", observed=False)
      .agg(
          appointments=("appointment_outcome", "size"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
)

previous_summary["no_show_rate"] = (
    previous_summary["no_shows"]
    / previous_summary["appointments"]
    * 100
)

display(previous_summary)

,appointments,no_shows,no_show_rate
previous_no_show_group,,,
0,2921,1271,43.51
1,1548,828,53.49
2+,531,324,61.02


### Validated finding

No-show rates increase as previous no-show history increases.

## 13. Analyzing Variables Together

Week 6 and Week 7 extended the analysis beyond individual variables by analyzing selected variables together.

The purpose was to determine whether combinations of factors produced different no-show patterns.

The validated combinations are:

1. Lead Time × Previous No-Show History
2. Lead Time × Reminder Status
3. Previous No-Show History × Reminder Status


In [13]:
# Calculate no-show rates by lead time and previous no-show history

lead_previous = (
    df.groupby(
        ["lead_time_group", "previous_no_show_group"],
        observed=False
    )
    .agg(
        appointments=("appointment_outcome", "size"),
        no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
    )
)

lead_previous["no_show_rate"] = (
    lead_previous["no_shows"]
    / lead_previous["appointments"]
    * 100
)

display(lead_previous)

appointments  no_shows  no_show_rate
lead_time_group previous_no_show_group                                      
0–7             0                                372        81         21.77
                1                                190        63         33.16
                2+                                78        34         43.59
8–14            0                                376       119         31.65
                1                                157        50         31.85
                2+                                69        33         47.83
15–30           0                                763       293         38.40
                1                                419       196         46.78
                2+                               151        87         57.62
31–45           0                                742       366         49.33
                1                                406       242         59.61
                2+                               110        69         62.73
46–60           0                                668       412         61.68
                1                                376       277         73.67
                2+                               123       101         82.11

### Validated finding

No-show rates increase across longer booking lead times, and the pattern is also higher among patients with more previous no-show history.

The highest validated combination was the 46–60 day lead-time group with 2+ previous no-shows.

In [14]:
# Calculate no-show rates by lead time and reminder status

lead_reminder = (
    df.groupby(
        ["lead_time_group", "reminder_sent"],
        observed=False
    )
    .agg(
        appointments=("appointment_outcome", "size"),
        no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
    )
)

lead_reminder["no_show_rate"] = (
    lead_reminder["no_shows"]
    / lead_reminder["appointments"]
    * 100
)

display(lead_reminder)

appointments  no_shows  no_show_rate
lead_time_group reminder_sent                                      
0–7             No                      167        50         29.94
                Yes                     473       128         27.06
8–14            No                      161        56         34.78
                Yes                     441       146         33.11
15–30           No                      378       176         46.56
                Yes                     955       400         41.88
31–45           No                      358       198         55.31
                Yes                     900       479         53.22
46–60           No                      302       222         73.51
                Yes                     865       568         65.66

### Validated finding

No-show rates increased across longer booking lead-time groups for both reminder-status groups.

Within the 46–60 day group, the validated no-show rate was 73.51% where no reminder was recorded compared with 65.66% where a reminder was recorded.


In [15]:
# Calculate no-show rates by previous history and reminder status

previous_reminder = (
    df.groupby(
        ["previous_no_show_group", "reminder_sent"],
        observed=False
    )
    .agg(
        appointments=("appointment_outcome", "size"),
        no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
    )
)

previous_reminder["no_show_rate"] = (
    previous_reminder["no_shows"]
    / previous_reminder["appointments"]
    * 100
)

display(previous_reminder)


appointments  no_shows  no_show_rate
previous_no_show_group reminder_sent                                      
0                      No                      790       359         45.44
                       Yes                    2131       912         42.80
1                      No                      428       242         56.54
                       Yes                    1120       586         52.32
2+                     No                      148       101         68.24
                       Yes                     383       223         58.22

### Validated finding

No-show rates were higher among patients with more previous no-show history across the reminder-status groups.

For patients with 2+ previous no-shows, the validated no-show rate was 68.24% where no reminder was recorded compared with 58.22% where a reminder was recorded.

## 14. Supporting Variables

Supporting analyses are retained because they were validated during Week 7.

In [16]:
# Create distance groups
distance_bins = [-np.inf, 5, 10, 20, np.inf]
distance_labels = [
    "0–5 km",
    "6–10 km",
    "11–20 km",
    "20+ km"
]

df["distance_group"] = pd.cut(
    df["distance_to_clinic_km"],
    bins=distance_bins,
    labels=distance_labels
)

distance_summary = (
    df.groupby("distance_group", observed=False)
      .agg(
          appointments=("appointment_outcome", "size"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
)

distance_summary["no_show_rate"] = (
    distance_summary["no_shows"]
    / distance_summary["appointments"]
    * 100
)

display(distance_summary)


,appointments,no_shows,no_show_rate
distance_group,,,
0–5 km,1156,537,46.45
6–10 km,1692,787,46.51
11–20 km,1669,825,49.43
20+ km,393,227,57.76


### Validated finding

The Week 7 validation showed that no-show rates differed across distance groups, with the 20+ km group recording a higher observed rate than the 0–5 km group.

Distance is treated as supporting context rather than a primary analytical driver.

In [17]:
waiting_summary = (
    df.groupby("appointment_outcome")["waiting_time_minutes"]
      .agg(["count", "mean"])
      .rename(columns={
          "count": "appointments",
          "mean": "average_waiting_time"
      })
)

display(waiting_summary)

,appointments,average_waiting_time
appointment_outcome,,
Attended,2293,24.29
Cancelled,261,23.21
No-Show,2386,24.20


In [18]:
# Review no-show rate by age group

age_summary = (
    df.groupby("age_group", observed=False)
      .agg(
          appointments=("appointment_outcome", "size"),
          no_shows=("appointment_outcome", lambda x: (x == "No-Show").sum())
      )
)

age_summary["no_show_rate"] = (
    age_summary["no_shows"]
    / age_summary["appointments"]
    * 100
)

display(age_summary)


,appointments,no_shows,no_show_rate
age_group,,,
18-24,564,283,50.18
25-34,783,397,50.70
35-44,819,396,48.35
45-54,793,381,48.05
55-64,800,406,50.75
65+,1241,560,45.12


## 15. Final Validated Findings

The following findings were reproduced from the cleaned dataset and are consistent with the validated Week 7 analysis:

1. Longer booking lead times were associated with higher observed no-show rates.
2. Patients with more previous no-show history recorded higher observed no-show rates.
3. Analyzing lead time together with previous no-show history showed the highest observed rates among combinations involving longer lead times and greater previous no-show history.
4. Analyzing lead time together with reminder status showed higher observed no-show rates at longer lead times, with differences between reminder-status groups.
5. Analyzing previous no-show history together with reminder status showed higher observed no-show rates among patients with greater previous no-show history.
6. Distance provided supporting variation in observed no-show rates.
7. Waiting time showed very similar average values between attended and no-show appointments.
8. Age-group differences provided additional descriptive context but were not treated as the primary analytical driver.

In [19]:
## Final validated findings table

final_findings = pd.DataFrame({
    "Area": [
        "Booking Lead Time",
        "Previous No-Show History",
        "Lead Time × Previous History",
        "Lead Time × Reminder",
        "Previous History × Reminder",
        "Distance",
        "Waiting Time",
        "Age Group"
    ],
    "Validated Finding": [
        "No-show rates increased across longer booking lead-time groups.",
        "No-show rates increased as previous no-show history increased.",
        "Higher observed no-show rates occurred when longer lead times were combined with greater previous no-show history.",
        "No-show rates increased across longer lead-time groups for both reminder-status groups.",
        "Higher observed no-show rates occurred among patients with greater previous no-show history across reminder-status groups.",
        "Observed no-show rates differed across distance groups.",
        "Average waiting times were very similar for attended and no-show appointments.",
        "Observed no-show rates differed across age groups."
    ],
    "Status": ["Validated"] * 8
})

display(final_findings)


,Area,Validated Finding,Status
0,Booking Lead Time,No-show rates increased across longer booking ...,Validated
1,Previous No-Show History,No-show rates increased as previous no-show hi...,Validated
2,Lead Time × Previous History,Higher observed no-show rates occurred when lo...,Validated
3,Lead Time × Reminder,No-show rates increased across longer lead-tim...,Validated
4,Previous History × Reminder,Higher observed no-show rates occurred among p...,Validated
5,Distance,Observed no-show rates differed across distanc...,Validated
6,Waiting Time,Average waiting times were very similar for at...,Validated
7,Age Group,Observed no-show rates differed across age gro...,Validated


## 16.  Additional Segment Validation for Data Science Integration

During Week 7, the Data Science track identified weaker model performance for:

- Specialist Consultation appointments
- Patients aged 65+

To provide additional descriptive context for these model limitations, targeted segment-level checks are conducted.

These checks do not represent a new primary analytics scope. They are supplementary analyses intended to determine whether existing analytical variables provide useful context within the identified segments.

The analysis focuses on variables already used in the HealthConnect analytics work:

- Booking Lead Time
- Previous No-Show History
- Reminder Status

The purpose is descriptive. These analyses do not establish causality or determine why model performance is weaker in either segment.

### 16.1. Isolating the 65+ Patient Segment

The analysis is restricted to patients in the `65+` age group. This allows the other variables to be examined specifically within the segment identified by the Data Science track.

In [20]:
# Filter to patients aged 65+
age_65 = df[df["age_group"] == "65+"].copy()

print(f"65+ segment size: {len(age_65):,}")

65+ segment size: 1,241


In [21]:
age_65["appointment_outcome"].value_counts()

appointment_outcome
Attended     605
No-Show      560
Cancelled     76
Name: count, dtype: int64

In [22]:
# Calculate 65+ no-show rate

age_65_no_show_rate = (
    age_65["appointment_outcome"].eq("No-Show").mean() * 100
)

print(f"65+ no-show rate: {age_65_no_show_rate:.2f}%")
print(f"65+ total appointments: {len(age_65):,}")


65+ no-show rate: 45.12%
65+ total appointments: 1,241


### Interpretation

Among patients aged 65+, the observed no-show rate was 45.12% based on valid appointments.

This provides the baseline no-show rate for the 65+ segment and serves as the reference point for examining how no-show rates vary across booking lead time, previous no-show history, and reminder status.

### 16.2. 65+ Patients by Booking Lead Time

Within the 65+ patient segment, no-show rates were calculated across the existing booking lead-time groups.

The age group is held constant at 65+, while booking lead time is examined against the appointment outcome.

This identifies whether observed no-show rates vary across different booking lead-time groups among patients aged 65+.

In [23]:
age_65_valid = df[df["age_group"] =="65+"]
age65_lead_time = (
    age_65_valid
    .groupby("lead_time_group", observed=False)["appointment_outcome"]
    .apply(lambda x: x.eq("No-Show").mean() * 100)
    .round(2)
    .reset_index(name="no_show_rate")
)

age65_lead_time

,lead_time_group,no_show_rate
0,0–7,29.12
1,8–14,26.11
2,15–30,39.31
3,31–45,56.63
4,46–60,62.09


### Interpretation

Among patients aged 65+, no-show rates varied across booking lead-time groups. The rate was 29.12% for appointments booked 0–7 days in advance and 26.11% for 8–14 days. It increased to 39.31 for 15–30 days, 56.63% for 31–45 days, and 62.09% for 46–60 days.

The results show a substantially higher observed no-show rate among 65+ patients with longer booking lead times, particularly from 31 days onward. This provides descriptive evidence of a notable booking lead-time pattern within the 65+ segment and adds context to the Data Science finding concerning weaker model performance for this age group.

### 16.3. 65+ Patients by Previous No-Show History

Within the 65+ patient segment, no-show rates were calculated across the existing previous no-show history groups.

This examines whether observed no-show rates vary according to previous no-show history among patients aged 65+.

In [24]:
age65_previous = (
    age_65_valid
    .groupby("previous_no_show_group", observed=False)["appointment_outcome"]
    .apply(lambda x: x.eq("No-Show").mean() * 100)
    .round(2)
    .reset_index(name="no_show_rate")
)

age65_previous

,previous_no_show_group,no_show_rate
0,0,37.72
1,1,53.81
2,2+,61.07


### Interpretation

Among patients aged 65+, the observed no-show rate increased across previous no-show history groups. Patients with no previous no-shows had a no-show rate of 37.72%, compared with 53.81% among patients with one previous no-show and 61.07% among patients with two or more previous no-shows.

This indicates that previous no-show history is associated with different observed no-show rates within the 65+ segment. The pattern provides additional descriptive context for the Data Science finding by showing that appointment history also differentiates no-show outcomes within this age group.

In [25]:
age65_reminder = (
    age_65_valid
    .groupby("reminder_sent")["appointment_outcome"]
    .apply(lambda x: x.eq("No-Show").mean() * 100)
    .round(2)
    .reset_index(name="no_show_rate")
)

age65_reminder

,reminder_sent,no_show_rate
0,No,46.04
1,Yes,44.78


### Interpretation

Among patients aged 65+, the observed no-show rate was 46.04% for appointments where no reminder was sent and 44.78% where a reminder was sent.

This represents a 1.26 percentage-point difference in observed no-show rates between the two reminder groups. The difference provides additional descriptive context for the 65+ segment, although it is smaller than the variation observed across booking lead-time and previous no-show history groups.

## 16.4 Overall 65+ Segment Interpretation

The additional analysis of patients aged 65+ identified observable differences in no-show rates across booking lead time, previous no-show history, and reminder status.

The strongest variation was observed across booking lead time, where the no-show rate increased from 26.11% for appointments booked within 14 days to 62.09% for appointments booked 46–60 days in advance. Previous no-show history also showed a clear pattern, with the rate increasing from 37.72% among patients with no previous no-shows to 61.07% among those with two or more previous no-shows. Reminder status showed a smaller difference, with no-show rates of 46.04% for patients without a reminder and 44.78% for those who received one.

These findings provide additional descriptive evidence about appointment behavior within the 65+ segment and therefore add context to the Data Science finding that the Week 7 model showed weaker performance for patients aged 65+.

The analysis does not establish causality and does not by itself explain why the model performed less strongly for this segment. It provides observed segment-level patterns that can be considered alongside the Data Science modelling results.

### 16.5 Specialist Consultation — Additional Analytics Context

Specialist Consultation was identified by the Data Science track as a segment where Week 7 model performance was weaker.

Appointment type was not part of the primary Week 8 Data Analytics analytical scope. Therefore, no new Specialist Consultation analysis is being presented here as an established Analytics finding.

## 17. Final Dashboard Validation

The Power BI dashboard developed and refined during the previous project stages remains the primary interactive visualization deliverable.

Week 8 validation confirms that the dashboard is consistent with the final analytical calculations.

The validation covers:

- KPI values;
- appointment outcome figures;
- lead-time findings;
- previous no-show history;
- combined-variable visuals;
- distance analysis;
- slicers;
- visual calculations; and
- consistency with the cleaned dataset.

The dashboard was not treated as a separate analytical source. Its figures were checked against the underlying analytical calculations.

## 18. Business Insights

### 18.1 Long booking lead time

No-show rates increased across the lead-time groups, reaching 67.69% for appointments booked 46–60 days ahead.

### 18.2 Previous no-show history

Repeated previous no-shows were associated with higher observed no-show rates.

### 18.3 Combined segmentation

Analyzing lead time and previous no-show history together showed higher observed rates in combinations such as 46–60 days with 2+ previous no-shows.

### 18.4 Reminder context

Reminder status showed differences in observed no-show rates across several groups. These are descriptive associations and do not prove that reminders caused the difference.

### 18.5 Supporting variables

Distance showed some observed variation, while waiting time showed little overall separation. Small groups require cautious interpretation.

## 19. Actionable Recommendations

1. **Use risk-based appointment monitoring.**  
   Use lead time and previous no-show history as monitoring signals.

2. **Review far-ahead appointments.**  
   Appointments booked substantially in advance can be monitored more closely.

3. **Use previous no-show history in support workflows.**  
   Repeated previous no-shows can inform appropriate administrative follow-up.

4. **Monitor reminder coverage.**  
   Maintain and monitor reminder coverage without interpreting observed differences as causal evidence.

5. **Continue dashboard monitoring.**  
   Use Power BI for ongoing monitoring of appointment patterns.

6. **Support future predictive modelling.**  
   The validated descriptive findings can provide context for future predictive work.

These are proposed actions. This notebook does not claim that the recommendations have already been implemented.

## 20. Final Conclusion

The Week 8 Analytics work confirms and packages the validated HealthConnect findings developed through Weeks 5–7.

The evidence shows that booking lead time and previous no-show history are useful descriptive segmentation variables, and analyzing them together provides additional context for identifying groups with higher observed no-show rates.

The Python notebook provides reproducible analytical evidence, while the Power BI dashboard provides interactive decision-support communication.

The Analytics component is ready for final integration and presentation, subject to completion of the actual HC-POD collaboration record and remaining final submission activities.